# 13.2 Automatic Speech Recognition

ASR turns acoustic evidence into words by combining frame-level probabilities, temporal structure, and language constraints.

Spectrogram or MFCC frames provide acoustic evidence, softmax converts logits to probabilities, and dynamic programming decodes the transcript users experience.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1301)


def sine_wave(freq, duration, fs, amplitude=1.0, phase=0.0):
    n = np.arange(int(round(duration * fs)))
    x = amplitude * np.sin(2.0 * np.pi * freq * n / fs + phase)
    return x


def chirp_wave(start_freq, end_freq, duration, fs, amplitude=1.0):
    n = np.arange(int(round(duration * fs)))
    t = n / fs
    slope = (end_freq - start_freq) / duration
    phase = 2.0 * np.pi * (start_freq * t + 0.5 * slope * t * t)
    x = amplitude * np.sin(phase)
    return x


def rms(x):
    return float(np.sqrt(np.mean(np.square(x))))


def add_noise_for_snr(x, snr_db, seed):
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(size=len(x))
    target_noise_rms = rms(x) / (10.0 ** (snr_db / 20.0))
    noise = noise / (rms(noise) + 1e-12)
    y = x + target_noise_rms * noise
    return y


def frame_signal(x, n_fft, hop):
    if len(x) < n_fft:
        pad_width = n_fft - len(x)
        x = np.pad(x, (0, pad_width))
    starts = np.arange(0, len(x) - n_fft + 1, hop)
    frames = np.stack([x[start:start + n_fft] for start in starts])
    return frames


def stft_mag(x, n_fft, hop):
    frames = frame_signal(x, n_fft, hop)
    window = np.hanning(n_fft)
    spectrum = np.fft.rfft(frames * window[None, :], axis=1)
    magnitude = np.abs(spectrum)
    return magnitude


def hz_to_mel(freq):
    return 2595.0 * np.log10(1.0 + freq / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def mel_filterbank(fs, n_fft, n_mels):
    freqs = np.linspace(0.0, fs / 2.0, n_fft // 2 + 1)
    mel_edges = np.linspace(hz_to_mel(0.0), hz_to_mel(fs / 2.0), n_mels + 2)
    hz_edges = mel_to_hz(mel_edges)
    bank = np.zeros((n_mels, len(freqs)))
    for band in range(n_mels):
        left = hz_edges[band]
        center = hz_edges[band + 1]
        right = hz_edges[band + 2]
        rising = (freqs - left) / max(center - left, 1e-12)
        falling = (right - freqs) / max(right - center, 1e-12)
        bank[band] = np.maximum(0.0, np.minimum(rising, falling))
    return bank


def dct_matrix(n):
    rows = []
    for k in range(n):
        row = []
        for b in range(n):
            value = math.cos(math.pi * k * (2 * b + 1) / (2 * n))
            row.append(value)
        rows.append(row)
    return np.array(rows)


def build_audio_ladder(fs=8000):
    d1 = sine_wave(440.0, 0.45, fs)
    d2 = 0.65 * sine_wave(440.0, 0.50, fs) + 0.35 * sine_wave(660.0, 0.50, fs)
    d3_clean = 0.60 * sine_wave(440.0, 0.55, fs) + 0.35 * sine_wave(880.0, 0.55, fs)
    d3 = add_noise_for_snr(d3_clean, 12.0, 1303)
    d4 = np.concatenate([
        sine_wave(330.0, 0.20, fs),
        chirp_wave(360.0, 900.0, 0.25, fs),
        0.80 * sine_wave(550.0, 0.20, fs),
        0.65 * sine_wave(720.0, 0.15, fs),
    ])
    d5_clean = np.concatenate([
        0.70 * sine_wave(300.0, 0.25, fs),
        chirp_wave(320.0, 980.0, 0.35, fs),
        0.55 * sine_wave(440.0, 0.25, fs) + 0.25 * sine_wave(880.0, 0.25, fs),
        0.70 * sine_wave(660.0, 0.30, fs),
        chirp_wave(900.0, 420.0, 0.25, fs),
    ])
    d5 = add_noise_for_snr(d5_clean, 5.0, 1305)
    ladder = [
        {"name": "D1 single synthetic sine", "x": d1, "fs": fs, "targets": [440.0], "complexity": 1},
        {"name": "D2 two-tone command", "x": d2, "fs": fs, "targets": [440.0, 660.0], "complexity": 2},
        {"name": "D3 noisy two-tone", "x": d3, "fs": fs, "targets": [440.0, 880.0], "complexity": 3},
        {"name": "D4 chirp multi-segment", "x": d4, "fs": fs, "targets": [330.0, 550.0, 720.0], "complexity": 4},
        {"name": "D5 longer noisier phrase", "x": d5, "fs": fs, "targets": [300.0, 440.0, 660.0], "complexity": 5},
    ]
    return ladder


def dominant_frequencies(x, fs, n_fft=512, hop=128, count=3):
    mag = stft_mag(x, n_fft, hop)
    mean_mag = np.mean(mag, axis=0)
    mean_mag[0] = 0.0
    order = np.argsort(mean_mag)[::-1]
    freqs = order[:count] * fs / n_fft
    return freqs


def nearest_error(predicted, targets):
    predicted = np.array(predicted)
    errors = []
    for target in targets:
        error = float(np.min(np.abs(predicted - target)))
        errors.append(error)
    return float(np.mean(errors))


## The concept, built once (D1): frame evidence

For a frame classifier, logits become probabilities with

$$p(c\mid x_t)=\frac{e^{z_c}}{\sum_{c'}e^{z_{c'}}}, \qquad \ell_t=-\log p(c_t\mid x_t).$$

The lesson logits $[1.2,0.3,-0.7]$ demonstrate max-shifted normalization.

In [ ]:

def softmax(logits):
    shifted = logits - np.max(logits)
    evidence = np.exp(shifted)
    probs = evidence / np.sum(evidence)
    return probs, evidence

lesson_logits = np.array([1.2, 0.3, -0.7])
lesson_probs, lesson_evidence = softmax(lesson_logits)
lesson_ce = -np.log(lesson_probs[0])

assert np.round(lesson_evidence, 3).tolist() == [1.0, 0.407, 0.15]
assert np.round(lesson_probs, 3).tolist() == [0.643, 0.261, 0.096]
assert round(float(lesson_ce), 3) == 0.442

print("evidence", np.round(lesson_evidence, 3).tolist())
print("probabilities", np.round(lesson_probs, 3).tolist())
print("cross entropy", round(float(lesson_ce), 3))


For sequence decoding, a transition model carries duration over time:

$$\alpha_t(j)=B_j(x_t)\sum_i\alpha_{t-1}(i)A_{ij}.$$

The decoder below combines acoustic emissions, transitions, optional language scores, and a duration penalty instead of labeling each frame independently.

In [ ]:

def decode_asr(frames, emissions, transitions, lm_scores=None, lm_weight=0.0, duration_bias=0.0):
    n_frames, n_states = emissions.shape
    log_emit = np.log(np.maximum(emissions, 1e-12))
    log_trans = np.log(np.maximum(transitions, 1e-12))
    if lm_scores is None:
        lm_scores = np.zeros(n_states)
    score = np.full((n_frames, n_states), -np.inf)
    back = np.zeros((n_frames, n_states), dtype=int)
    score[0] = log_emit[0] + lm_weight * lm_scores
    for t in range(1, n_frames):
        for j in range(n_states):
            stay_bonus = np.zeros(n_states)
            stay_bonus[j] = duration_bias
            candidates = score[t - 1] + log_trans[:, j] + stay_bonus
            back[t, j] = int(np.argmax(candidates))
            score[t, j] = candidates[back[t, j]] + log_emit[t, j] + lm_weight * lm_scores[j]
    states = [int(np.argmax(score[-1]))]
    for t in range(n_frames - 1, 0, -1):
        states.append(int(back[t, states[-1]]))
    states = states[::-1]
    collapsed = []
    for state in states:
        if not collapsed or collapsed[-1] != state:
            collapsed.append(state)
    return states, collapsed, score

def forward_probabilities(emissions, transitions):
    alpha = np.array([1.0, 0.0, 0.0])
    history = []
    for emit in emissions:
        alpha = emit * (alpha @ transitions)
        alpha = alpha / np.sum(alpha)
        history.append(alpha.copy())
    return np.array(history)

transitions = np.array([
    [0.72, 0.25, 0.03],
    [0.05, 0.70, 0.25],
    [0.02, 0.18, 0.80],
])
emissions = np.array([
    [0.75, 0.20, 0.05],
    [0.15, 0.65, 0.20],
    [0.248545, 0.242950, 1.000000],
])
alpha_history = forward_probabilities(emissions, transitions)
wer = (1 + 1 + 1) / 5

assert np.round(alpha_history[-1], 3).tolist() == [0.173, 0.329, 0.497]
assert round(float(wer), 3) == 0.6

print("final normalized forward probabilities", np.round(alpha_history[-1], 3).tolist())
print("WER example", wer)


## The dataset ladder

Each rung is NumPy-synthesized audio plus known token labels: a single sine token, a two-token command, noisy audio, a chirped utterance, and a longer repeated-token command.

In [ ]:

TOKENS = ["low", "mid", "high"]
TOKEN_FREQS = np.array([440.0, 660.0, 880.0])

def make_asr_ladder(fs=8000):
    base = build_audio_ladder(fs)
    labels = [
        [0],
        [0, 1],
        [0, 2],
        [0, 1, 2],
        [0, 1, 1, 2, 0],
    ]
    ladder = []
    for rung, label in zip(base, labels):
        item = dict(rung)
        item["label_ids"] = label
        item["transcript"] = [TOKENS[i] for i in label]
        ladder.append(item)
    return ladder

asr_ladder = make_asr_ladder()

for rung in asr_ladder:
    print(rung["name"])
    print("  shape", rung["x"].shape)
    print("  transcript", " ".join(rung["transcript"]))
    print("  preview", np.round(rung["x"][:6], 3).tolist())


## Run the SAME method across D1-D5

The same softmax-emission plus dynamic-programming ASR decoder is applied to every rung, and the metric is transcript exact-match accuracy.

In [ ]:

def emissions_from_audio(x, fs, token_freqs, n_fft=256, hop=64):
    mag = stft_mag(x, n_fft, hop)
    freqs = np.arange(n_fft // 2 + 1) * fs / n_fft
    columns = []
    for freq in token_freqs:
        weights = np.exp(-0.5 * ((freqs - freq) / 55.0) ** 2)
        score = mag @ weights
        columns.append(score)
    logits = np.stack(columns, axis=1)
    logits = logits / np.maximum(np.std(logits, axis=1, keepdims=True), 1e-6)
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exps / np.sum(exps, axis=1, keepdims=True)
    return probs

def collapse_to_tokens(states):
    collapsed = []
    for state in states:
        if not collapsed or collapsed[-1] != state:
            collapsed.append(state)
    return collapsed

asr_rows = []
base_transitions = np.array([
    [0.80, 0.17, 0.03],
    [0.07, 0.80, 0.13],
    [0.05, 0.15, 0.80],
])

for rung in asr_ladder:
    probs = emissions_from_audio(rung["x"], rung["fs"], TOKEN_FREQS)
    states, decoded_ids, score = decode_asr(None, probs, base_transitions, lm_weight=0.0, duration_bias=0.15)
    trimmed = decoded_ids[:len(rung["label_ids"])]
    exact = int(trimmed == rung["label_ids"])
    asr_rows.append({"rung": rung, "probs": probs, "states": states, "decoded": trimmed, "metric": exact})

print("rung | exact transcript accuracy | decoded")
for row in asr_rows:
    decoded_words = [TOKENS[i] for i in row["decoded"]]
    print(row["rung"]["name"], "|", row["metric"], "|", " ".join(decoded_words))


## Results visualization

The closing figure has two parts: spectrograms with decoded token timelines, then accuracy vs complexity.

In [ ]:

fig, axes = plt.subplots(5, 2, figsize=(10, 12))
acc = []

for idx, row in enumerate(asr_rows):
    rung = row["rung"]
    mag = stft_mag(rung["x"], rung["fs"], 256, 64)
    axes[idx, 0].imshow(20.0 * np.log10(mag.T + 1e-6), origin="lower", aspect="auto", cmap="magma")
    axes[idx, 0].set_title(rung["name"] + " spectrogram")
    axes[idx, 1].plot(row["states"], drawstyle="steps-post")
    axes[idx, 1].set_ylim(-0.5, 2.5)
    axes[idx, 1].set_yticks([0, 1, 2])
    axes[idx, 1].set_yticklabels(TOKENS)
    axes[idx, 1].set_title("decoded token timeline")
    acc.append(row["metric"])

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), acc, marker="o")
plt.xlabel("complexity rung")
plt.ylabel("exact transcript accuracy")
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.show()


## Pitfall on D5: a language model can overpower acoustics

A fluent prior may force the wrong command even when emissions point to the noisy signal.  Lower the LM weight and add a duration bias so acoustics and persistence matter again.

In [ ]:

d5_row = asr_rows[-1]
d5_probs = d5_row["probs"]
fluent_wrong_prior = np.array([0.0, 1.7, 2.0])
wrong_states, wrong_ids, wrong_score = decode_asr(None, d5_probs, base_transitions, fluent_wrong_prior, lm_weight=2.0, duration_bias=0.0)
fixed_states, fixed_ids, fixed_score = decode_asr(None, d5_probs, base_transitions, fluent_wrong_prior, lm_weight=0.25, duration_bias=0.25)
wrong_trimmed = wrong_ids[:len(asr_ladder[-1]["label_ids"])]
fixed_trimmed = fixed_ids[:len(asr_ladder[-1]["label_ids"])]
wrong_accuracy = int(wrong_trimmed == asr_ladder[-1]["label_ids"])
fixed_accuracy = int(fixed_trimmed == asr_ladder[-1]["label_ids"])

print("wrong LM-heavy decoded", [TOKENS[i] for i in wrong_trimmed], wrong_accuracy)
print("fixed decoded", [TOKENS[i] for i in fixed_trimmed], fixed_accuracy)


## Evaluate it + Practice

- Metric: transcript exact-match accuracy across D1-D5
- No-skill baseline: guess the most common/simple output and compare against the ladder metric.
- Cheap sanity check: the lesson logits should normalize to probabilities [0.643, 0.261, 0.096] and CE 0.442
- Ablation: turn off transitions and duration bias; repeated phones should fragment or collapse incorrectly
- Failure signals: high frame accuracy but wrong words, over-fluent commands, or WER worsening on noisy D5

Practice prompts:

**Practice.** Change the acoustic weight by scaling logits and inspect D5's transcript.

**Practice.** Add a fourth token frequency and update the transition matrix.

**Practice.** Compute WER for the decoded command instead of exact match.